## API DATAJUD

In [1]:
# ================================================================
# 00) Imports
# ================================================================
import os, json, uuid, math, logging, requests
from time import perf_counter
from datetime import datetime
from zoneinfo import ZoneInfo
from concurrent.futures import ThreadPoolExecutor, as_completed
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

from pyspark.sql import SparkSession
from pyspark.sql.functions import col, from_utc_timestamp, current_timestamp, to_timestamp
from delta.tables import DeltaTable

# ================================================================
# 01) Configurações principais
# ================================================================
spark = SparkSession.builder.getOrCreate()
spark.conf.set("spark.sql.session.timeZone", "America/Sao_Paulo")

# Tabelas
TABELA_FONTE_PROCESSOS = "DOL_arqs_auxiliares.lista_processos_datajud"
TABELA_API_DESTINO     = "DOL_arqs_auxiliares.api_datajud2"

# Execução
DATA_EXEC    = datetime.now(ZoneInfo("America/Sao_Paulo")).strftime("%Y%m%d_%H%M%S")
EXECUCAO_ID  = str(uuid.uuid4())

# Diretórios
DIR_BASE       = "Files/DATAJUD_V2/resultado-api"
DIR_LOTE_BASE  = f"{DIR_BASE}/lotes/{DATA_EXEC}_{EXECUCAO_ID}"
CHECKPOINT     = f"/lakehouse/default/{DIR_BASE}/checkpoints/lotes_processados.json"

# API CNJ
API_KEY       = "ApiKey cDZHYzlZa0JadVREZDJCendQbXY6SkJlTzNjLV9TRENyQk1RdnFKZGRQdw=="
API_BASE_URL  = "https://api-publica.datajud.cnj.jus.br/api_publica_tjsp/_search"

# Performance
BATCH_SIZE    = 500
MAX_WORKERS   = 5
TIMEOUT_SECS  = 30
RETRIES       = 3
BACKOFF       = 2

StatementMeta(, fa80e429-9cfe-4632-8968-1df5069c5968, 3, Finished, Available, Finished)

In [2]:
# ================================================================
# 02) Logging
# ================================================================
for h in logging.root.handlers[:]:
    logging.root.removeHandler(h)

logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s")

StatementMeta(, fa80e429-9cfe-4632-8968-1df5069c5968, 4, Finished, Available, Finished)

In [3]:
# ================================================================
# 03) Funções utilitárias
# ================================================================
def build_session():
    """Cria sessão Requests com retry e backoff"""
    retry = Retry(
        total=RETRIES,
        backoff_factor=BACKOFF,
        status_forcelist=[429, 500, 502, 503, 504],
        allowed_methods=["POST"]
    )
    s = requests.Session()
    s.mount("https://", HTTPAdapter(max_retries=retry))
    s.mount("http://",  HTTPAdapter(max_retries=retry))
    return s

def build_headers():
    return {
        "Authorization": f"{API_KEY}",
        "Content-Type": "application/json",
        "User-Agent": "DATAJUD-Fabric/1.0"
    }

def build_payload(numero_processo: str) -> dict:
    return {"query": {"match": {"numeroProcesso": numero_processo}}}

def fetch_processo(numero_processo: str, session: requests.Session) -> dict:
    """Consulta um processo na API pública do DataJud."""
    try:
        payload = json.dumps(build_payload(numero_processo))
        resp = session.post(API_BASE_URL, headers=build_headers(), data=payload, timeout=TIMEOUT_SECS)
        resp.raise_for_status()
        data = resp.json()
        resultados = []
        for hit in data.get("hits", {}).get("hits", []):
            registro = hit.get("_source", {})
            if registro:
                registro["id"] = registro.get("id")
                resultados.append(registro)
        return {
            "ok": True,
            "numeroProcesso": numero_processo,
            "payload": resultados[0] if resultados else {},
            "data_execucao": datetime.now(ZoneInfo("America/Sao_Paulo")).strftime("%Y-%m-%d %H:%M:%S")
        }
    except Exception as e:
        return {
            "ok": False,
            "numeroProcesso": numero_processo,
            "erro": str(e),
            "data_execucao": datetime.now(ZoneInfo("America/Sao_Paulo")).strftime("%Y-%m-%d %H:%M:%S")
        }

def save_batch_ndjson(registros: list, path_jsonl: str):
    """Salva lista de dicts como NDJSON em Files/..."""
    os.makedirs(os.path.dirname(f"/lakehouse/default/{path_jsonl}"), exist_ok=True)
    with open(f"/lakehouse/default/{path_jsonl}", "w", encoding="utf-8") as f:
        for r in registros:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")

# ------------------------------------------------
# Controle de checkpoint global (retomada)
# ------------------------------------------------
def obter_checkpoint():
    """Lê o último lote processado (se existir)."""
    if os.path.exists(CHECKPOINT):
        try:
            with open(CHECKPOINT, "r", encoding="utf-8") as f:
                meta = json.load(f)
                return meta.get("ultimo_lote_finalizado", -1)
        except Exception as e:
            logging.warning(f"Falha ao ler checkpoint: {e}")
    return -1

def atualizar_checkpoint(lote_num):
    """Atualiza checkpoint global após processar um lote."""
    os.makedirs(os.path.dirname(CHECKPOINT), exist_ok=True)
    meta = {
        "ultimo_lote_finalizado": lote_num,
        "data_registro": datetime.now(ZoneInfo("America/Sao_Paulo")).strftime("%Y-%m-%d %H:%M:%S"),
        "execucao_id": EXECUCAO_ID,
        "data_exec": DATA_EXEC
    }
    with open(CHECKPOINT, "w", encoding="utf-8") as f:
        json.dump(meta, f, ensure_ascii=False, indent=2)
    logging.info(f"Checkpoint atualizado: lote {lote_num:,}")


StatementMeta(, fa80e429-9cfe-4632-8968-1df5069c5968, 5, Finished, Available, Finished)

In [4]:
# ================================================================
# 04) Carregar lista de processos
# ================================================================
logging.info("Carregando lista de processos da tabela Silver...")
inicio = perf_counter()

df_processos = spark.table(TABELA_FONTE_PROCESSOS).select("Processo").distinct()
processos = [r["Processo"] for r in df_processos.collect()]
total_processos = len(processos)

logging.info(f"Total de processos a consultar nesta execução: {total_processos:,}")
logging.info(f"Execução: {DATA_EXEC} | EXECUCAO_ID: {EXECUCAO_ID}")

os.makedirs(f"/lakehouse/default/{DIR_LOTE_BASE}", exist_ok=True)
session = build_session()

StatementMeta(, fa80e429-9cfe-4632-8968-1df5069c5968, 6, Finished, Available, Finished)

2025-10-14 19:19:09,068 - INFO - Carregando lista de processos da tabela Silver...
2025-10-14 19:21:42,446 - INFO - Total de processos a consultar nesta execução: 29,318,063
2025-10-14 19:21:42,447 - INFO - Execução: 20251014_161858 | EXECUCAO_ID: 65187e95-4fcf-4ad0-a8ed-5e04dd22492e


In [5]:
# ================================================================
# 05) Execução em lotes com retomada automática
# ================================================================
n_lotes = math.ceil(total_processos / BATCH_SIZE)
ultimo_lote_finalizado = obter_checkpoint()
lote_inicial = ultimo_lote_finalizado + 1

sucesso, erros = 0, 0
logging.info(f"Retomando processamento a partir do lote {lote_inicial:,} de {n_lotes:,} disponíveis.")

for i in range(lote_inicial, n_lotes):
    ini = i * BATCH_SIZE
    fim = min((i + 1) * BATCH_SIZE, total_processos)
    pedaco = processos[ini:fim]
    if not pedaco:
        continue

    t0 = perf_counter()
    resultados = []

    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as pool:
        futures = {pool.submit(fetch_processo, p, session): p for p in pedaco}
        for fut in as_completed(futures):
            resultados.append(fut.result())

    ok = sum(1 for r in resultados if r["ok"])
    nok = len(resultados) - ok
    sucesso += ok
    erros += nok

    path_jsonl = f"{DIR_LOTE_BASE}/lote_{i:06d}.jsonl"
    save_batch_ndjson(resultados, path_jsonl)
    atualizar_checkpoint(i)

    logging.info(
        f"Lote {i+1}/{n_lotes} | OK: {ok:,} | Erros: {nok:,} | "
        f"Tempo: {perf_counter() - t0:.2f}s | Path: {path_jsonl}"
    )

logging.info(f"Consulta finalizada. Sucesso: {sucesso:,} | Erros: {erros:,} | Tempo total: {perf_counter() - inicio:.2f}s")

StatementMeta(, fa80e429-9cfe-4632-8968-1df5069c5968, 7, Submitted, Running, Running)

2025-10-14 19:21:54,716 - INFO - Retomando processamento a partir do lote 0 de 58,637 disponíveis.
2025-10-14 19:22:03,664 - INFO - Checkpoint atualizado: lote 0
2025-10-14 19:22:03,664 - INFO - Lote 1/58637 | OK: 500 | Erros: 0 | Tempo: 8.95s | Path: Files/DATAJUD_V2/resultado-api/lotes/20251014_161858_65187e95-4fcf-4ad0-a8ed-5e04dd22492e/lote_000000.jsonl
2025-10-14 19:22:15,456 - INFO - Checkpoint atualizado: lote 1
2025-10-14 19:22:15,456 - INFO - Lote 2/58637 | OK: 500 | Erros: 0 | Tempo: 11.79s | Path: Files/DATAJUD_V2/resultado-api/lotes/20251014_161858_65187e95-4fcf-4ad0-a8ed-5e04dd22492e/lote_000001.jsonl
2025-10-14 19:22:26,281 - INFO - Checkpoint atualizado: lote 2
2025-10-14 19:22:26,287 - INFO - Lote 3/58637 | OK: 500 | Erros: 0 | Tempo: 10.83s | Path: Files/DATAJUD_V2/resultado-api/lotes/20251014_161858_65187e95-4fcf-4ad0-a8ed-5e04dd22492e/lote_000002.jsonl
2025-10-14 19:22:39,248 - INFO - Checkpoint atualizado: lote 3
2025-10-14 19:22:39,249 - INFO - Lote 4/58637 | OK: 5

In [ ]:
# ================================================================
# 06) Ler JSON → DataFrame consolidado
# ================================================================
logging.info("Lendo JSON de lotes para Spark DataFrame...")

df_ndjson = spark.read.json(f"{DIR_LOTE_BASE}/*.jsonl")

df_erros = df_ndjson.filter(~col("ok"))
qtd_erros = df_erros.count()
if qtd_erros > 0:
    logging.warning(f"Erros registrados em {qtd_erros:,} processos.")
    df_erros.write.format("delta").mode("append").saveAsTable("DOL_arqs_auxiliares.api_datajud_erros")

df_ok = df_ndjson.filter(col("ok")).drop("ok")

df_api = (
    df_ok
    .withColumn("data_execucao", to_timestamp(col("data_execucao"), "yyyy-MM-dd HH:mm:ss"))
    .withColumn("data_ingestao", from_utc_timestamp(current_timestamp(), "America/Sao_Paulo"))
    .withColumn("payload_json", col("payload"))
    .withColumn("id", col("payload.id"))
    .withColumn("dataHoraUltimaAtualizacao", to_timestamp(col("payload.dataHoraUltimaAtualizacao")))
    .select("id", "numeroProcesso", "dataHoraUltimaAtualizacao", "payload_json", "data_execucao", "data_ingestao")
)

df_api = df_api.filter(col("id").isNotNull()).dropDuplicates(["id"])

In [ ]:
# ================================================================
# 07) Merge incremental 
# ================================================================
if not spark._jsparkSession.catalog().tableExists(TABELA_API_DESTINO):
    (df_api
        .repartition(256, "id")
        .write.format("delta")
        .mode("overwrite")
        .saveAsTable(TABELA_API_DESTINO))
    logging.info(f"Tabela criada: {TABELA_API_DESTINO}")
else:
    tgt = DeltaTable.forName(spark, TABELA_API_DESTINO)
    (tgt.alias("t")
        .merge(df_api.alias("s"), "t.id = s.id")
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute())
    logging.info(f"Merge concluído na tabela: {TABELA_API_DESTINO}")

In [ ]:
# ================================================================
# 08) Resumo final de execução
# ================================================================
tempo_total = perf_counter() - inicio_total
ultimo_cp = obter_checkpoint()

logging.info("------------------------------------------------")
logging.info("RESUMO FINAL DE EXECUÇÃO")
logging.info("------------------------------------------------")
logging.info(f"Data/Hora início: {DATA_EXEC}")
logging.info(f"Execução ID:      {EXECUCAO_ID}")
logging.info(f"Último lote salvo: {ultimo_cp:,}")
logging.info(f"Total de processos consultados: {total_processos:,}")
logging.info(f"Lotes processados: {ultimo_cp + 1:,} / {n_lotes:,}")
logging.info(f"Sucessos: {sucesso:,} | Erros: {erros:,}")
logging.info(f"Tempo total: {tempo_total/60:.2f} min ({tempo_total:.2f}s)")
logging.info("------------------------------------------------")
logging.info("Execução concluída.")